# 🎧 MagL+X: PyTorch Loss Sonification (Runnable Notebook)

This notebook demonstrates how to:
- Train a simple PyTorch model
- Extract per-sample losses
- Convert them into audio
- Stream real-time sound using MagL+X concepts

---


In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch numpy sounddevice

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import sounddevice as sd


## 🔊 Synth Engine

In [ ]:
SR = 44100
BLOCK = 512

class Synth:
    def sine(self, freq, amp, frames):
        t = np.arange(frames) / SR
        return amp * np.sin(2 * np.pi * freq * t)

    def distribution_sound(self, dist, frames):
        dist = np.array(dist)
        if dist.sum() > 0:
            dist = dist / dist.sum()
        signal = np.zeros(frames)
        for i, w in enumerate(dist):
            signal += self.sine(220 * (i + 1), w, frames)
        return signal

    def xent_sound(self, p, q, frames):
        eps = 1e-9
        H = -np.sum(p * np.log(q + eps))
        base = self.distribution_sound(p, frames)
        mod = np.sin(2 * np.pi * (2 + H*10) * np.arange(frames) / SR)
        noise = np.random.randn(frames) * (H * 0.1)
        return base * (1 + 0.5 * mod) + noise

synth = Synth()

## 🎛 Live Audio Engine

In [ ]:
class LiveEngine:
    def __init__(self):
        self.current_audio_fn = None

    def callback(self, outdata, frames, time_info, status):
        if self.current_audio_fn:
            audio = self.current_audio_fn(frames)
        else:
            audio = np.zeros(frames)
        outdata[:] = audio.reshape(-1,1)

    def start(self):
        self.stream = sd.OutputStream(callback=self.callback,
                                      channels=1,
                                      samplerate=SR,
                                      blocksize=BLOCK)
        self.stream.start()

    def stop(self):
        self.stream.stop()

engine = LiveEngine()

## 🧠 Loss → Audio Mapping

In [ ]:
def loss_to_audio_fn(loss_tensor):
    values = loss_tensor.detach().cpu().flatten().numpy()
    values = np.clip(values, 0, None)
    if values.sum() > 0:
        values = values / values.sum()

    def audio_fn(frames):
        return synth.distribution_sound(values, frames)

    return audio_fn

class CrossEntropyTracker:
    def __init__(self):
        self.prev = None

    def to_audio_fn(self, loss_tensor):
        curr = loss_tensor.detach().cpu().flatten().numpy()
        curr = np.clip(curr, 0, None)
        if curr.sum() > 0:
            curr = curr / curr.sum()

        if self.prev is None:
            self.prev = curr
            return lambda frames: np.zeros(frames)

        prev = self.prev
        self.prev = curr

        def audio_fn(frames):
            return synth.xent_sound(prev, curr, frames)

        return audio_fn

tracker = CrossEntropyTracker()

## 🧪 Toy Dataset

In [ ]:
# Simple synthetic classification data
X = torch.randn(200, 10)
y = (X.sum(dim=1) > 0).long()

## 🤖 Model

In [ ]:
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 2)
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(reduction='none')

## ▶️ Training with Live Audio

In [ ]:
engine.start()

for epoch in range(5):
    logits = model(X)
    loss = criterion(logits, y)

    optimizer.zero_grad()
    loss.mean().backward()
    optimizer.step()

    # choose either:
    # audio_fn = loss_to_audio_fn(loss)
    audio_fn = tracker.to_audio_fn(loss)

    engine.current_audio_fn = audio_fn

    print(f"Epoch {epoch}, loss = {loss.mean().item():.4f}")

engine.stop()